# 07 · Dynamically built effective Hamiltonians

No fixed model types: spaces + symmetry group + grammar terms are compiled, certified and fitted.
Offline and CPU-only.

1. Groups and characters (counts, spin statistics, Jahn-Teller activity).
2. Product-space compile: methane ν3 rovibrational Coriolis, exact.
3. Operator dictionary with Molien-count certificates, and contact-transformation redundancy
   (Watson's reduction recovered from the algebra alone).
4. `autobuild`: the loop proposes, filters and accepts terms on evidence.
5. Line strengths from the built model.
6. Proposers: amortized network and language model (offline stand-in).
7. CH₄⁺: T₂ ⊗ (e + t₂) Jahn-Teller couplings from the dictionary.

In [ ]:
import numpy as np
import riss_drucker as rd
from riss_drucker import build as B, algebra as A, autobuild as AB, symmetry as S
td = S.get_group("Td")
print({J: td.level_counts(2 * J) for J in range(5)})
print("CH4 spin weights", td.spin_statistics([0.5] * 4), " JT-active for F2:", td.jahn_teller_active("F2"))

In [ ]:
spec = B.BuildSpec([B.RotSpace(), B.VibSpace([B.Mode("3", "F2")], polyads=[0, 1])], "Td",
                   ["w3: n3", "B: J2", "z: sym(i*Jz a3x+ a3y) + h.c."], J=list(range(5)))
bm = B.compile_build(spec)
print(len(bm.certificates), "certificates passed")
th = {"w3": 3019.0, "B": 5.24, "z": 3 * 2 * 5.24 * 0.05}          # -2 B zeta J.l with zeta = 0.05
for b, (w, _) in zip(bm.blocks, bm.eig(th)):
    if b.polyad == 1 and b.two_j <= 4:
        print(f"J={b.two_j // 2} {b.irrep:3s}", np.round(w, 3))

In [ ]:
rs = B.BuildSpec([B.RotSpace()], "D2", [], J=list(range(9)))
d = A.enumerate_terms(rs, A.multidegrees(rs, rot=(2, 6)))
print([(x["degree"], x["count"], x["predicted"]) for x in d.by_degree])
bm = B.compile_build(rs.with_terms(d.terms))
rng = np.random.default_rng(1)
th = dict(zip(bm.param_names, [3.0, 1.5, 1.0] + list(1e-4 * rng.standard_normal(6)) + list(1e-7 * rng.standard_normal(10))))
r = A.redundancy(bm, th, A.multidegrees(rs, rot=(3, 5)))
print("terms", r["n_terms"], "removable", r["redundant"], "identifiable", r["identifiable_max"])

In [ ]:
q = d.terms
truth = B.compile_build(rs.with_terms(q[:3] + q[3:8]))
tt = dict(zip(truth.param_names, [1.5, 0.5, 1.5] + list(-3e-5 * (1 + np.abs(rng.standard_normal(5))))))
data = [AB.LevelDatum(b.two_j / 2, b.irrep, i, e + 1e-4 * rng.standard_normal(), 1e-4)
        for b, (w, _) in zip(truth.blocks, truth.eig(tt)) for i, e in enumerate(w) if b.two_j <= 12]
res = AB.autobuild(B.BuildSpec([B.RotSpace()], "D2", q[:3], J=list(range(7))), data,
                   dict(zip(truth.param_names[:3], [1.5, 0.5, 1.5])),
                   [AB.DictionaryProposer(lambda s: A.multidegrees(s, rot=(4, 4)))],
                   redundancy_degrees=A.multidegrees(rs, rot=(3, 3)), new_term_sigma=1e-3)
for s in res.steps:
    print(s.accepted, f"{s.term:32s}", s.reason)
print("chi2/dof", round(res.fit.reduced_chi2, 2), res.certificates[-1]["name"], res.certificates[-1]["passed"])

In [ ]:
bm = B.compile_build(B.BuildSpec([B.RotSpace()], "D2", ["A: Jz Jz", "Bx: Jx Jx", "Cy: Jy Jy"], J=list(range(6))))
th = {"A": 3.0, "Bx": 1.5, "Cy": 1.0}
lines = B.line_strengths(bm, th, {"z": 1.0})
Q = B.intensities(bm, th, lines, T=10.0)
for ln in sorted(lines, key=lambda l: -l["intensity"])[:5]:
    print(round(ln["wavenumber"], 4), round(ln["strength"], 4), f"{ln['intensity']:.3e}")

In [ ]:
from riss_drucker.ml import structure
from riss_drucker.llm import LLMProposer
modes = [B.Mode("1", "A", 2), B.Mode("2", "A", 1)]
vspec = B.BuildSpec([B.VibSpace(modes, polyads=[0, 1, 2, 3, 4])], "C1", ["w1: n1", "w2: n2"])
cands = ["k: a1+ a2 a2 + h.c.", "x11: n1 n1", "x22: n2 n2", "x12: n1 n2"]
ds = structure.simulate(vspec, cands, lambda r: {"w1": r.uniform(2900, 3100), "w2": r.uniform(1450, 1550)},
                        {"k": 30.0, "x11": 10.0, "x22": 5.0, "x12": 8.0}, 1500, n_levels=5, seed=0)
prop = structure.AmortizedProposer(ds)
truth = B.compile_build(vspec.with_terms(vspec.terms + [cands[0]]))
obs = [AB.LevelDatum(0, b.irrep, i, e, 0.01, b.polyad) for b, (w, _) in zip(truth.blocks, truth.eig({"w1": 3010.0, "w2": 1500.0, "k": 25.0})) for i, e in enumerate(w)]
print({k.split(":")[0]: round(v, 3) for k, v in prop.probabilities(obs).items()})
fake_llm = lambda prompt: "DK: Jz Jz Jz Jz\nnot a term"
print(LLMProposer(fake_llm).propose({"spec": rs.with_terms(q[:3])}))

In [ ]:
sp = [B.ElecSpace([("1", "F2")]), B.VibSpace([B.Mode("2", "E"), B.Mode("4", "F2")], nmax=4)]
jt = B.BuildSpec(sp, "Td", [])
en = A.enumerate_terms(jt, A.multidegrees(jt, vib=(1, 1)))
print(en.by_degree[0]["count"], "linear JT couplings:", en.terms)